In [78]:
import pyspark
import re
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.appName('BigD_OurD').getOrCreate()
sc = spark.sparkContext

In [79]:
########################
# Act 3: Partitioning  #
########################

# Initialization of the whole RDD
def csvFilter(line):
    out = []
    if "\"" not in line:
        return line.split(",")
    else:
        start = 0
        indices = [match.start() for match in re.finditer(r",", line)]
        while len(indices) > 10:
            indices.pop(1)
        indices.append(-1)
        for end in indices:
             out.append(line[start:end])
             start = end+1
        return out

game_rdd = sc.textFile("vgsales.csv").map(csvFilter)
header = game_rdd.first()
game_rdd = game_rdd.filter(lambda x: x != header)

In [80]:
###################################
# Genre to Global Sale Comparison #
#    by Keiron Mirandilla         #
###################################

# Keep genre(5th) and global sales(last) as a tuple w/ Hash Partitioning
# Get sum of global sales and sort by highest
genre_sales = game_rdd.map(lambda x: (x[4], float(x[-1]))).partitionBy(4).persist()
summed_sales = genre_sales.reduceByKey(lambda a,b: a+b).sortBy(lambda x: x[1], ascending=False)

top_genres = summed_sales.take(3)

In [81]:
print("Top 3 Genres by Global Sales:")
for i in range(3):
    print(f"  {i+1}. {top_genres[i][0]} with {top_genres[i][1]:.2f} million sales.")

Top 3 Genres by Global Sales:
  1. Action with 1748.75 million sales.
  2. Sports with 1330.73 million sales.
  3. Shooter with 1035.85 million sales.


In [83]:
#Nash A. Mapula

# Part 1: RDD Strategy (Video Game Sales)
# Load the file as an RDD
vgsales_rdd = sc.textFile("/content/vgsales.csv")

# Remove header and split by comma
header = vgsales_rdd.first()
data_rdd = vgsales_rdd.filter(lambda line: line != header).map(lambda line: line.split(","))

# Range Partitioning by Year (Index 3)
# map it to (Year, Full_Row) so Spark can use Year as the partition key
keyed_rdd = data_rdd.map(lambda x: (x[3], x))

# Use 5 partitions based on the Year ranges
partitioned_rdd = keyed_rdd.partitionBy(5)

print(f"Number of partitions: {partitioned_rdd.getNumPartitions()}")

Number of partitions: 5


In [87]:
##############################
# Act 4: Data Preprocessing  #
#        and DataFrames      #
##############################

# Initialization and pre-processing of dataframe and schema
headers = [
    ("Show ID", StringType(), False),
    ("Type", StringType(), False),
    ("Title", StringType(), False),
    ("Director", StringType(), True),
    ("Cast", StringType(), True),
    ("Country", StringType(), True),
    ("Date Added", StringType(), True),
    ("Release Year", IntegerType(), False),
    ("Rating", StringType(), True),
    ("Duration", StringType(), True),
    ("Listed In", StringType(), False),
    ("Description", StringType(), False)
]

netflix_schema = StructType([
    StructField(*field) for field in headers
])
netflix_df = spark.read.csv("netflix_titles.csv", header=True, schema=netflix_schema)

# Get headers with missing values and
kv_nulls = {key[0]: "N/A" for key in headers if key[2]}
netflix_df = netflix_df.fillna(kv_nulls)

In [88]:
netflix_df.select("*").show()

+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|Show ID|   Type|               Title|            Director|                Cast|             Country|        Date Added|Release Year|Rating| Duration|           Listed In|         Description|
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|     Kirsten Johnson|                 N/A|       United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|                 N/A|Ama Qamata, Khosi...|        South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglan